---
title: "숙제 3"
subtitle: "데이터과학 입문"
author: "원중호"
date: today
date-format: "MMMM YYYY"
institute: 서울대학교 통계학과
format:
  html:
      embed-resources: true

mainfont: "Noto Sans Korean Light"
---

```{r setup, include=FALSE}
knitr::opts_chunk$set(echo = TRUE, eval = TRUE, warning = FALSE, message = FALSE, fig.width=6, fig.height=4, out.width = "70%", fig.align = "center", python.reticulate = TRUE)  

options(knitr.table.format = "html")
reticulate::use_condaenv("introds", required = TRUE)
```

## 지시사항

제출마감 2026-06-15 23:00

1.	R과 Python을 모두 사용하여 사용된 코드와 데이터랭글링 절차, 분석결과를 설명한다. 두 언어의 분석결과가 차이가 있으면 그 이유를 설명한다.
2.  [Quarto Markdown](https://quarto.org/docs/authoring/markdown-basics.html)을 사용한다. 제공된 숙제 `.qmd` 파일에 본인의 답안을 "답안" 절에 추가하여 제출한다. Quarto Markdown은 RStudio 또는 Visual Studio Code에 [Quarto Extension](https://marketplace.visualstudio.com/items?itemName=quarto.quarto)을 추가하여 컴파일, 다른 문서 형식으로 변환할 수 있다. 
3.  R의 `reticulate` 패키지를 사용하면 하나의 `.qmd` 파일 안에서 R과 Python을 동시에 사용할 수 있다. 이때 다음 문법을 사용하여 두 언어 코드를 탭으로 구분한다.  숙제 `.qmd` 파일은 `reticulate`을 사용하도록 준비되어 있다.

````
::: {.panel-tabset}

## R

```{{r}}
R code
```

## Python

```{{python}}
Python code
```

:::

````

3.  `.qmd`를 컴파일하여 생성된 `.html` 파일을 함께 저장소에 제출한다.
4.  함께 제공된 `student.yml`을 함께 작성하여 저장소에 제출한다.

## 평가 기준

1.  재현성: 제출된 저장소의 `.qmd` 파일을 컴파일하여 함께 제출된 `.html` 파일과 동일한 결과가 나와야 한다.
2.	분석의 정확성: 분석은 올바른 기술적 세부 사항을 포함하여 수행되어야 한다.
3.	보고서의 전반적인 품질: 데이터 가공 및 분석 결과가 명확하고 자세하게 설명되어야 한다.
4.	코드의 전반적인 품질: 코드는 체계적으로 정리되어 있어야 하며, 가독성을 높이기 위해 적절한 주석이 포함되어야 한다.

#### **늦게 제출된 과제물은 받지 않는다.**

# 1부  교과서 연습문제

## 문제 1-1

1. MDSR 10장 연습문제 10.6.6

### 답안(연습문제 10.6.6)

#### **데이터 랭글링 절차**

**R**: `NHANES` 패키지의 `NHANES` 데이터에서 20세 이상 관측치를 필터링한 후,
`SmokeNow`와 `Smoke100` 변수를 결합하여 현재 정기적 흡연 여부를 나타내는 이진
변수 `SmokeCurrent`를 생성하였다. `SmokeNow == "Yes"`이면 1, `SmokeNow == "No"` 또는 `Smoke100 == "No"`이면 0으로 리코딩하였으며, 나머지 결측값은 분석에서 제외하였다. 이후 예측변수(`Age`, `Gender`, `Race1`, `Education`, `HHIncomeMid`, `BMI`, `PhysActive`, `Alcohol12PlusYr`, `Depressed`, `SleepHrsNight`)를 포함한 완전한 케이스만 선택하였다 (N = 5,898).
**Python**: Polars를 사용하여 CSV 파일에서 동일한 데이터 랭글링 절차를 수행하였다. `pl.when().then().otherwise()` 구문으로 `SmokeCurrent`를 생성하고, `drop_nulls()` 로 결측값을 제거한 후 statsmodels 적합을 위해 Pandas DataFrame으로 변환하였다. `Depressed` 변수의 기준 범주가 Python의 경우 알파벳순으로 `Most`로 설정되어 R의 `None`과 달라지므로, `C(Depressed, Treatment('None'))`으로 명시적으로 지정하여 두 언어의 결과를 일치시켰다.

------------------------------------------------------------------------

::: panel-tabset
## R

```{r}
library(tidyverse)
library(NHANES)
nhanes_smoke <- NHANES %>%
  filter(Age >= 20) %>%
  # SmokeNow의 NA 중 Smoke100 == "No"인 경우를 0으로 리코딩
  mutate(
    SmokeCurrent = case_when(
      SmokeNow == "Yes"  ~ 1L,   # 현재 정기적 흡연자
      SmokeNow == "No"   ~ 0L,   # 과거 흡연자 (현재 비흡연)
      Smoke100 == "No"   ~ 0L,   # 비흡연자 (100개비 미만)
      TRUE               ~ NA_integer_  # 나머지 NA 제거
    ),
    SmokeCurrent = factor(SmokeCurrent, levels = c(0, 1),
                          labels = c("No", "Yes"))
  ) %>%
  # 예측변수 + 결과변수만 선택 후 완전한 케이스만 사용
  select(SmokeCurrent, Age, Gender, Race1, Education,
         HHIncomeMid, BMI, PhysActive,
         Alcohol12PlusYr, Depressed, SleepHrsNight) %>%
  drop_na()

# 로지스틱 회귀 모델 적합
smoke_model <- glm(
  SmokeCurrent ~ Age + Gender + Race1 + Education +
    HHIncomeMid + BMI + PhysActive +
    Alcohol12PlusYr + Depressed + SleepHrsNight,
  data   = nhanes_smoke,
  family = binomial
)

summary(smoke_model)
```

## Python

In [ ]:
import polars as pl
from plotnine import *
import pandas as pd
import statsmodels.formula.api as smf

NHANES = pl.read_csv('data/NHANES.csv',
    schema_overrides={
        "Age": pl.Float64, 
        "BMI": pl.Float64
    },
    null_values=["NA", " ", "null"] 
)

# 1. 데이터 랭글링 절차
nhanes_smoke = (
    NHANES
    .filter(pl.col("Age") >= 20)
    .with_columns(
        pl.when(pl.col("SmokeNow") == "Yes").then(1)
        .when(pl.col("SmokeNow") == "No").then(0)
        .when(pl.col("Smoke100") == "No").then(0)
        .otherwise(None)
        .alias("SmokeCurrent")
    )
    .select([
        "SmokeCurrent", "Age", "Gender", "Race1", "Education",
        "HHIncomeMid", "BMI", "PhysActive", 
        "Alcohol12PlusYr", "Depressed", "SleepHrsNight"
    ])
    .drop_nulls()
)

# 2. 모델 적합을 위해 Pandas로 변환 
nhanes_smoke_pd = nhanes_smoke.to_pandas()

# 3. 로지스틱 회귀 모델 적합

formula = """
    SmokeCurrent ~ Age + C(Gender) + C(Race1) + C(Education) + 
    HHIncomeMid + BMI + C(PhysActive) + C(Alcohol12PlusYr) + 
    C(Depressed, Treatment('None')) + SleepHrsNight
"""

smoke_model = smf.logit(formula=formula, data=nhanes_smoke_pd).fit()

# 4. 결과 요약 출력
print(smoke_model.summary())

:::

------------------------------------------------------------------------

#### **결과해석**
20세 이상 성인 5,898명을 대상으로 현재 정기적 흡연 여부(`SmokeCurrent`)를
예측하는 로지스틱 회귀 모델을 적합하였다 (현재 흡연자 비율: 20.1%).
모델의 AIC는 4,962이며, Null deviance 대비 Residual deviance가 5,909에서
4,926으로 감소하여 모델이 유의미한 설명력을 가짐을 확인하였다.

유의한 예측변수($\alpha = 0.05$)는 다음과 같다.

- **나이 (`Age`)**: 나이가 증가할수록 현재 흡연 가능성이 유의하게 감소하였다
  ($\hat\beta = -0.026$, $p < 0.001$).
- **성별 (`Gender`)**: 남성이 여성에 비해 흡연 가능성이 유의하게 높았다
  ($\hat\beta = 0.175$, $p = 0.018$).
- **인종 (`Race1`)**: Black을 기준으로 Hispanic($\hat\beta = -0.487$)과
  Mexican($\hat\beta = -1.118$)은 흡연 가능성이 유의하게 낮았으나,
  White와 Other는 유의한 차이가 없었다.
- **교육 수준 (`Education`)**: 8학년 이하를 기준으로 고등학교 졸업 이상부터
  흡연 가능성이 유의하게 낮아졌으며, 대학 졸업자에서 가장 강한 음의 효과를
  보였다($\hat\beta = -1.737$, $p < 0.001$).
- **가구 소득 (`HHIncomeMid`)**: 소득이 높을수록 흡연 가능성이 유의하게
  감소하였다($\hat\beta = -7.4 \times 10^{-6}$, $p < 0.001$).
- **BMI**: BMI가 높을수록 흡연 가능성이 유의하게 감소하였다
  ($\hat\beta = -0.046$, $p < 0.001$).
- **신체활동 (`PhysActive`)**: 신체활동을 하는 사람이 그렇지 않은 사람보다
  흡연 가능성이 유의하게 낮았다($\hat\beta = -0.635$, $p < 0.001$).
- **음주 여부 (`Alcohol12PlusYr`)**: 음주자가 비음주자보다 흡연 가능성이
  유의하게 높았으며($\hat\beta = 1.351$, $p < 0.001$), 모든 예측변수 중
  가장 강한 양의 효과를 보였다.
- **우울감 (`Depressed`)**: 우울감이 없는 경우(None)에 비해 Several
  ($\hat\beta = 0.559$)과 Most($\hat\beta = 0.358$) 모두 흡연 가능성이
  유의하게 높았다.
- **수면 시간 (`SleepHrsNight`)**: 수면 시간이 길수록 흡연 가능성이 유의하게
  감소하였다($\hat\beta = -0.169$, $p < 0.001$).

#### **두 언어의 결과 비교 및 차이점**
R과 Python의 모든 회귀 계수, 표준오차, z값, p값이 완전히 일치하였다.
단, `Depressed` 변수의 기준 범주 설정에서 차이가 발생하였다. R은 factor
level 순서에 따라 `None`을 기준으로 설정하는 반면, Python `statsmodels`의
`C()` 함수는 알파벳순으로 `Most`를 기준으로 설정한다. 이를 해결하기 위해
Python에서 `C(Depressed, Treatment('None'))`으로 기준 범주를 명시적으로
지정하여 두 언어의 결과를 일치시켰다.

# 2부  데이터 분석 실무

### 분석 관련 공통 지침

1.	관측단위(observational unit)는 `playerID`와 `yearID`의 고유한 조합으로 한다. 즉, 데이터프레임의 각 행은 한 선수의 특정 연도에 해당해야 하고(예: 2019년 류현진), 한 선수의 특정 연도가 두 번 이상 나타나서는 안 된다. 이적을 한 경우 원자료에서는 두 번 이상 나타날 수 있으므로 주의해야 한다.
2.	데이터 분석을 하는 중에 필요한 경우 pivoting으로 각 행이 한명의 선수에 해당하는 wide format data를 만들어서 연도간 비교를 하는 것은 허용한다.


## 문제 2-1

Lahman Package의 `Teams` 데이터프레임에서 코로나 시즌인 2020년을 제외한 2010년부터 2025년 사이의 데이터를 이용하여 다음 질문에 답하라. 

1.  MDSR Chapter 7 Iteration 에서 배운 Bill James의 공식을 변형한 다음 모형을 데이터에 적합하고, 모수 $k$의 점추정치와 신뢰구간을 구하라.
$$
  WPct = \frac{RS^k}{S^k+RA^k} = \frac{1}{1+(RA/RS)^k}
$$

2.  회귀계수 $\beta_1$이 위 모형의 $k$와 거의 같은 의미를 가지는 로지스틱 회귀 모형을 세우고 이를 데이터에 적합하라. 모수와 점추정치와 신뢰구간을 구하고 이를 1항의 결과와 비교하라. 

    *주의*: 절편이 없는 모형을 적합해야 함.
    *힌트 1*. 로짓은 $\log〖WPct/(1-WPct)$로 계산됨.
    *힌트 2*. 로짓의 역함수인 sigmoid는 $\frac{1}{1+e^{-x}}$로 계산됨.

3.  2항의 모형 적합 결과에 대한 다음 세가지 진단 중 최소 두가지 이상을 수행하여 모형적합이 잘 되었는지 확인하라.

    i.  Residual Deviance에 대한 해석 (카이제곱 분포와 비교) 
	  ii. Deviance residuals vs linear predictors ($\eta$) 산점도 
	  iii.  관측된 WPct와 모형에서 예측하는 WPct를 산점도 그래프로 비교

4.  `WPct`를 반응변수로, `log(RA)`와 `log(RS)`를 설명변수로 하는 절편이 없는 로지스틱선형회귀 모형을 적합하고 회귀계수들의 추정 결과를 a와 b항의 결과와 비교하라. (유사한 모형을 얻는지 여부 등)


### 답안(2-1-1)

#### **데이터 랭글링 절차**

**R**: `Lahman` 패키지의 `Teams` 데이터에서 2010년부터 2025년 사이 데이터를
필터링하고 2020년 코로나 시즌을 제외하였다. 승률(`WPct`)을 승수(`W`)를 경기수
(`G`)로 나누어 계산하였다. 이후 비선형 최소제곱법(`nls`)을 사용하여 피타고라스
승률 공식의 모수 $k$를 추정하였으며, 초기값은 야구 선행 연구에서 통용되는 2로
설정하였다. 신뢰구간은 `confint()`를 사용하여 산출하였다.

**Python**: `pylahman` 패키지가 2025년 데이터를 포함하지 않아 R의 `Lahman`
패키지에서 내보낸 `Teams.csv`를 Polars로 불러와 동일한 필터링 및 승률 계산을
수행하였다. `scipy.optimize.curve_fit`을 이용하여 최소제곱 추정을 수행하였으며,
공분산 행렬에서 표준오차를 추출한 후 t-분포를 사용하여 95% 신뢰구간을
계산하였다.

------------------------------------------------------------------------

::: panel-tabset
## R

```{r}
library(tidyverse)
library(Lahman)

# 데이터 랭글링
teams_filtered <- Teams %>%
  filter(yearID >= 2010, yearID <= 2025, yearID != 2020) %>%
  mutate(WPct = W / G)


# 비선형 최소제곱법(NLS) 모델 적합
# 야구에서 k값은 1.8~2.0 사이이므로 초기값(start)을 2로 설정
teams_fm <- nls(
  WPct ~ 1 / (1 + (RA/R)^k), 
  data = teams_filtered, 
  start = list(k = 2)
)

# 모수 k의 점추정치 확인
summary(teams_fm)

# 95% 신뢰구간 구하기
confint(teams_fm)
```

## Python

In [ ]:
import polars as pl
# import pylahman
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from scipy.stats import t

# pylahman 버전 업데이트 시 오류 발생으로 인해 
# Teams 데이터를 r Lahman package에서 들고옴
Teams = pl.read_csv(
    "data/Teams.csv",
    null_values=["NA", " ", "null"] )

# 2. 데이터 랭글링
teams_filtered = (
    Teams.filter(
        pl.col("yearID").is_between(2010, 2025), 
        pl.col("yearID") != 2020
    )
    .with_columns(
        WPct = pl.col("W") / pl.col("G"),
        Ratio = pl.col("RA") / pl.col("R")
    )
)

def pythagorean_model(ratio, k):
    return 1 / (1 + ratio**k)
  
# 3. 독립/종속 변수를 NumPy 배열로 변환
x_data = teams_filtered["Ratio"].to_numpy()
y_data = teams_filtered["WPct"].to_numpy()

# 4. curve_fit을 이용한 파라미터 추정 (초기값 p0=2.0)
popt, pcov = curve_fit(pythagorean_model, x_data, y_data, p0=[2.0])
k_est = popt[0]

# 5. 공분산 행렬에서 표준오차 추출
se = np.sqrt(np.diag(pcov))[0]

# 6. 95% 신뢰구간 계산 (t-분포 사용)
alpha = 0.05
dof = len(x_data) - 1
t_val = t.ppf(1.0 - alpha/2.0, dof)

ci_lower = k_est - t_val * se
ci_upper = k_est + t_val * se

# 7. 결과 출력
print(f"k 점추정치: {k_est:.4f}")
print(f"95% 신뢰구간: [{ci_lower:.4f}, {ci_upper:.4f}]")

:::

------------------------------------------------------------------------

#### **결과 해석**

2010년부터 2025년(2020년 제외) MLB 팀 데이터(n = 450)를 이용하여 피타고라스
승률 공식의 모수 $k$를 비선형 최소제곱법으로 추정하였다. 추정 결과
$\hat{k} = 1.752$ (95% CI: [1.695, 1.810])로 나타났으며, 이는 통계적으로
유의하였다($p < 0.001$). 야구에서 흔히 사용되는 $k = 2$ (단순 피타고라스
공식)나 Bill James의 개선 추정치 $k = 1.83$보다 다소 낮은 값으로, 해당 기간
MLB 데이터에서 팀 승률을 가장 잘 설명하는 득실점 지수는 약 1.75임을 의미한다.

#### **두 언어의 결과 비교 및 차이점**

동일한 데이터를 사용한 결과 R과 Python의 $\hat{k}$ 추정치(1.7524)가 완전히
일치하였다. 신뢰구간도 R([1.6948, 1.8101])과 Python([1.6947, 1.8100])이
반올림 오차 수준으로 일치하였다. 단, 신뢰구간 계산 방식에서 R의
`confint.nls`는 자유도를 잔차 자유도($n - p = 449$)로 사용하는 반면,
Python은 $n - 1 = 449$로 설정하였다. 이 문제에서는 모수가 $k$ 하나이므로
두 방식의 자유도가 동일하게 449가 되어 결과 차이가 발생하지 않았다.


### 답안(2-1-2)

#### **데이터 랭글링 절차**

**R**: 문제 2-1-1에서 생성한 `teams_filtered`에 득점/실점 로그 비율
(`log_ratio = log(R/RA)`)과 패수(`L = G - W`)를 추가하였다. 절편 없는
로지스틱 회귀 모형을 적합할 때, 승률과 같은 비율 데이터는 `cbind(W, L)`을
종속변수로 지정하는 것이 정석이다. 이는 각 팀의 경기 수가 달라 관측치마다
시행 횟수가 다른 이항 데이터임을 명시적으로 반영한다.

**Python**: `teams_filtered`에 `log_ratio`를 추가한 후 Pandas로 변환하였다.
`statsmodels`에서는 종속변수에 비율(`WPct`)을 넣고 `var_weights`에 총
경기 수(`G`)를 지정하면 R의 `cbind(W, L)` 방식과 동일하게 각 관측치의
시행 횟수를 가중치로 반영할 수 있다.

------------------------------------------------------------------------

::: panel-tabset
## R

```{r}
# R 데이터 랭글링
teams_logistic <- teams_filtered %>%
  mutate(
    log_ratio = log(R / RA),
    L = G - W
  )

# 절편 없는 로지스틱 회귀 모형 적합 (0 + 추가)
# 주의: R에서 승률 같은 비율 데이터를 이항 로지스틱으로 돌릴 때는 
# 종속변수에 cbind(성공, 실패)를 넣어주는 것이 정석입니다.
logit_model_r <- glm(
  cbind(W, L) ~ 0 + log_ratio, 
  data = teams_logistic, 
  family = binomial
)

# 모수 점추정치 확인
summary(logit_model_r)

# 95% 신뢰구간 확인
confint(logit_model_r)
```

## Python

In [ ]:
import polars as pl
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Python 데이터 랭글링 (이전 블록의 teams_filtered Polars DF 활용)
teams_logistic_py = (
    teams_filtered
    .with_columns(
        log_ratio = np.log(pl.col("R") / pl.col("RA"))
    )
)

teams_pd = teams_logistic_py.to_pandas()

# 절편 없는 로지스틱 회귀 모형 적합 (0 + 추가)
# Python statsmodels에서는 종속변수에 비율(WPct)을 넣고, 
# var_weights에 총 시행횟수(G)를 주면 R의 cbind(W, L)과 동일하게 작동합니다.
logit_model_py = smf.glm(
    formula="WPct ~ 0 + log_ratio",
    data=teams_pd,
    family=sm.families.Binomial(),
    var_weights=teams_pd["G"]
).fit()

# 모수 점추정치 확인
print(logit_model_py.summary())

# 95% 신뢰구간 확인
print(logit_model_py.conf_int(alpha=0.05))

:::

------------------------------------------------------------------------

#### **결과해석**
절편 없는 로지스틱 회귀 모형을 적합한 결과 $\hat{\beta}_1 = 1.753$
(95% CI: [1.663, 1.842])으로 추정되었으며, 이는 통계적으로 유의하였다
($p < 0.001$). 이 모형에서 $\log(R/RA)$의 회귀계수 $\beta_1$은 피타고라스
공식의 지수 $k$와 동일한 의미를 가진다. 로짓 변환을 통해 확인하면,
$\text{logit}(WPct) = \beta_1 \cdot \log(R/RA) = k \cdot \log(R/RA)$이고,
이를 역변환하면 $WPct = \frac{1}{1+(RA/R)^k}$으로 문제 2-1-1의 피타고라스
공식과 수학적으로 동일한 형태임을 알 수 있다.

2-1-1의 NLS 추정치 $\hat{k} = 1.752$ (95% CI: [1.695, 1.810])와 비교하면
점추정치가 거의 동일하나, 로지스틱 회귀의 신뢰구간([1.663, 1.842])이 다소
넓게 나타났다. 이는 두 방법이 서로 다른 추정 원리(비선형 최소제곱 vs.
최대가능도 추정)를 사용하기 때문이다.

#### **두 언어의 결과 비교 및 차이점**
R과 Python 모두 독립변수 $\log(R/RA)$에 대한 회귀계수와 표준오차, 신뢰구간이 동일하게 도출된다. 다만 비율 데이터(WPct)에 대해 이항 로지스틱 회귀를 적용할 때, R의 glm()은 종속변수 자리에 cbind(성공 횟수, 실패 횟수) 행렬을 입력받아 시행 횟수에 따른 가중치를 자연스럽게 처리하는 방식을 쓴다. 반면 Python의 statsmodels.formula.api.glm()은 종속변수에 비율(WPct)을 직접 명시하되, var_weights 파라미터에 경기 수(G)를 할당하여 통계적 분산을 보정한다는 점에서 문법적 차이가 존재한다.

R과 Python의 $\hat{\beta}_1$ 추정치(1.7527), 표준오차(0.046), z값(38.42),
Residual deviance(180.38), 95% 신뢰구간([1.663, 1.842]) 모두 완전히
일치하였다.

### 답안(2-1-3)

#### **데이터 랭글링 절차**

**R**: `logit_model_r` 객체에서 `fitted()`, `residuals()`, `predict()`를
사용하여 예측 승률(`fitted_WPct`), Deviance 잔차(`resid_dev`), 선형 예측값
(`eta`)을 `teams_logistic`에 추가하였다. Residual Deviance의 유의성은
`pchisq()`로 카이제곱 분포와 비교하였으며, 산점도는 `ggplot2`로 시각화하였다.

**Python**: `logit_model_py` 객체의 `fittedvalues`, `resid_deviance` 속성을
Polars DataFrame에 결합하였다. 선형 예측값(`eta`)은 `log_ratio`에 추정된
계수를 곱하여 직접 계산하였다. Residual Deviance의 유의성은
`scipy.stats.chi2.cdf()`로 평가하였으며, 산점도는 `plotnine`으로 시각화하였다.

------------------------------------------------------------------------

::: panel-tabset
## R

```{r}
library(tidyverse)
library(broom)

# 1. 모델 진단을 위한 데이터 추출 (fitted values, residuals 등)
teams_diag <- teams_logistic %>%
  mutate(
    fitted_WPct = fitted(logit_model_r),
    resid_dev = residuals(logit_model_r, type = "deviance"),
    eta = predict(logit_model_r, type = "link")
  )

# i. Residual Deviance 평가
res_dev <- summary(logit_model_r)$deviance
df_res <- summary(logit_model_r)$df.residual
p_value <- 1 - pchisq(res_dev, df_res)

cat("Residual Deviance:", res_dev, "\n")
cat("Degrees of Freedom:", df_res, "\n")
cat("P-value:", p_value, "\n\n")

# ii. Deviance residuals vs linear predictors 산점도
plot_ii <- ggplot(teams_diag, aes(x = eta, y = resid_dev)) +
  geom_point(alpha = 0.5, color = "blue") +
  geom_hline(yintercept = 0, linetype = "dashed", color = "red") +
  labs(
    title = "Deviance Residuals vs Linear Predictors",
    x = "Linear Predictor (eta)",
    y = "Deviance Residual"
  ) +
  theme_minimal()

print(plot_ii)

# iii. 관측된 WPct vs 예측된 WPct 산점도
plot_iii <- ggplot(teams_diag, aes(x = fitted_WPct, y = WPct)) +
  geom_point(alpha = 0.5, color = "darkgreen") +
  geom_abline(intercept = 0, slope = 1, linetype = "dashed", color = "red") +
  labs(
    title = "Observed vs Predicted WPct",
    x = "Predicted WPct",
    y = "Observed WPct (Actual)"
  ) +
  theme_minimal()

print(plot_iii)
```

## Python

In [ ]:
import polars as pl
import numpy as np
import scipy.stats as stats
from plotnine import *
import matplotlib.pyplot as plt

# 1. 모델 진단을 위한 데이터 추출
# statsmodels의 logit_model_py 객체에서 파생 변수들을 Polars로 결합
# (이전 단계에서 teams_logistic_py, logit_model_py, teams_pd가 정의되어 있다고 가정)
teams_diag = (
    teams_logistic_py
    .with_columns(
        fitted_WPct = pl.Series(logit_model_py.fittedvalues),
        resid_dev = pl.Series(logit_model_py.resid_deviance),
        # statsmodels에서 eta(linear predictor)는 X * beta로 계산
        eta = pl.col("log_ratio") * logit_model_py.params.iloc[0]
    )
)

# i. Residual Deviance 평가
res_dev = logit_model_py.deviance
df_res = logit_model_py.df_resid
p_value = 1 - stats.chi2.cdf(res_dev, df_res)

print(f"Residual Deviance: {res_dev:.4f}")
print(f"Degrees of Freedom: {df_res}")
print(f"P-value: {p_value:.4f}\n")

# ii. Deviance residuals vs linear predictors 산점도
plot_ii = (
    ggplot(teams_diag, aes(x="eta", y="resid_dev"))
    + geom_point(alpha=0.5, color="blue")
    + geom_hline(yintercept=0, linetype="dashed", color="red")
    + labs(
        title="Deviance Residuals vs Linear Predictors",
        x="Linear Predictor (eta)",
        y="Deviance Residual"
    )
    + theme_minimal()
)
plot_ii.show()

# iii. 관측된 WPct vs 예측된 WPct 산점도
plot_iii = (
    ggplot(teams_diag, aes(x="fitted_WPct", y="WPct"))
    + geom_point(alpha=0.5, color="darkgreen")
    + geom_abline(intercept=0, slope=1, linetype="dashed", color="red")
    + labs(
        title="Observed vs Predicted WPct",
        x="Predicted WPct",
        y="Observed WPct (Actual)"
    )
    + theme_minimal()
)
plot_iii.show()

:::

------------------------------------------------------------------------

#### **결과해석**

세 가지 진단을 통해 모형 적합도를 평가하였다.

**i. Residual Deviance (카이제곱 분포와 비교)**

Residual Deviance = 180.38, 자유도 = 449로, 귀무가설(모형이 데이터를 잘
설명함) 하에서 카이제곱 분포를 따른다. p-value = 1.000으로, Residual
Deviance가 자유도에 비해 매우 작아 과소산포(underdispersion) 경향이
있음을 의미한다. 이는 모형이 데이터를 충분히 잘 설명하고 있으며, 적합
부족(lack of fit)의 근거가 없음을 나타낸다. 

**ii. Deviance Residuals vs Linear Predictors 산점도**

잔차가 선형 예측값($\eta$) 전 범위에 걸쳐 0을 중심으로 무작위하게
분포하며, 뚜렷한 패턴이 관찰되지 않는다. 이는 모형의 체계적 오적합이
없음을 시사한다.

**iii. 관측값 vs 예측값 산점도**

관측된 승률과 모형이 예측한 승률이 기준선($y = x$) 주변에 밀집하여
분포하고 있어, 모형이 팀 승률을 전반적으로 잘 예측하고 있음을 확인할 수 있다.

세 진단 결과를 종합하면, 절편 없는 로지스틱 회귀 모형은 MLB 팀 승률
데이터에 잘 적합되었다고 판단할 수 있다.

#### **두 언어의 결과 비교 및 차이점**

R과 Python의 Residual Deviance(180.38), 자유도(449), p-value(1.000)가
완전히 일치하였다. 산점도 역시 동일한 데이터를 기반으로 하여 동일한
결과를 보였다. R은 `ggplot2`, Python은 `plotnine`을 사용하였으나 두
라이브러리 모두 Grammar of Graphics 체계를 따르므로 코드 구조가 거의
동일하다.

### 답안(2-1-4)

#### **데이터 랭글링 절차**

**R**: 문제 2-1-1에서 생성한 `teams_filtered`에 `log(R)`과 `log(RA)`를
추가하였다. 절편 없는 로지스틱 회귀 모형을 `cbind(W, L)`을 종속변수로
하여 적합하였다.

**Python**: `teams_filtered`에 `log_R`, `log_RA`를 추가한 후 Pandas로
변환하였다. 2-1-2와 동일하게 `var_weights=G`로 시행 횟수를 반영하여
절편 없는 로지스틱 회귀 모형을 적합하였다.

------------------------------------------------------------------------

::: panel-tabset
## R

```{r}
# R 데이터 랭글링: 기존 teams_filtered에 log_R, log_RA 추가
teams_logistic_sep <- teams_filtered %>%
  mutate(
    log_R = log(R),
    log_RA = log(RA),
    L = G - W
  )

# 절편 없는 로지스틱 선형회귀 모형 적합 (0 + log_R + log_RA)
logit_model_r_sep <- glm(
  cbind(W, L) ~ 0 + log_R + log_RA, 
  data = teams_logistic_sep, 
  family = binomial
)

# 회귀계수 추정치 및 요약 결과 확인
summary(logit_model_r_sep)

# 95% 신뢰구간 확인
confint(logit_model_r_sep)
```

## Python

In [ ]:
import polars as pl
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Python 데이터 랭글링: 기존 teams_filtered에 log_R, log_RA 추가
teams_logistic_py_sep = (
    teams_filtered
    .with_columns(
        log_R = np.log(pl.col("R")),
        log_RA = np.log(pl.col("RA"))
    )
)

teams_pd_sep = teams_logistic_py_sep.to_pandas()

# 절편 없는 로지스틱 선형회귀 모형 적합 (0 + log_R + log_RA)
logit_model_py_sep = smf.glm(
    formula="WPct ~ 0 + log_R + log_RA",
    data=teams_pd_sep,
    family=sm.families.Binomial(),
    var_weights=teams_pd_sep["G"]
).fit()

# 회귀계수 추정치 및 요약 결과 확인
print(logit_model_py_sep.summary())

# 95% 신뢰구간 확인
print(logit_model_py_sep.conf_int(alpha=0.05))

:::

------------------------------------------------------------------------

#### **결과해석**

$\log(R)$과 $\log(RA)$를 각각 설명변수로 분리하여 절편 없는 로지스틱
회귀를 적합한 결과, $\hat{\beta}_{log\_R} = 1.753$, $\hat{\beta}_{log\_RA}
= -1.753$으로 두 계수의 절댓값이 거의 동일하고 부호만 반대로 나타났다.

이는 2-1-2의 단일 계수 $\hat{\beta}_1 = 1.753$과 수학적으로 동치임을
보여준다. $\log(R/RA) = \log(R) - \log(RA)$이므로, $\beta \cdot \log(R/RA)$는
$\beta \cdot \log(R) + (-\beta) \cdot \log(RA)$와 같고, 실제로 두 계수의
추정치가 $1.753$과 $-1.753$으로 이를 확인할 수 있다. 따라서 2-1-1의
피타고라스 공식, 2-1-2의 단일 로그비율 모형, 그리고 본 모형은 모두
수학적으로 동일한 구조를 가진다.

한편 2-1-2와 비교하면 Residual Deviance는 180.38에서 180.24로 미세하게
감소하였으나, 자유도가 449에서 448로 하나 줄어 AIC는 2665.6에서 2667.5로
오히려 증가하였다. 이는 설명변수를 분리하여도 모형의 실질적인 설명력이
개선되지 않음을 나타내며, 두 계수가 사실상 동일한 정보를 담고 있음을
재확인한다.

#### **두 언어의 결과 비교 및 차이점**

R과 Python의 $\hat{\beta}_{log\_R}$(1.7527), $\hat{\beta}_{log\_RA}$(-1.7531),
Residual Deviance(180.24), 자유도(448), 95% 신뢰구간이 모두 일치하였다.

## 문제 2-2

`WPct`를 반응변수로, `logRS`, `logRA`, `H`, `X2B`, `X3B`, `HR`, `BB`, `SO`, `CS`, `HBP`, `SF`, `ERA`, `CG`, `SHO`, `IPouts`, `HA`, `HRA`, `BBA`, `SOA`, `E`, `DP`, `FP`, `SV`를 설명변수로 하는 절편항이 있는 로지스틱 회귀 모형을 적합하고 AIC를 기준으로 하는 단계별(stepwise) 변수선택을 적용하라. 변수선택 후 남은 변수들을 모두 모형에 남길지 일부를 제거할지 다시 판단하라. 최종적으로 선택된 모형을 문제1의 모형과 비교하라. 

### 답안(2-2)

#### **데이터 랭글링 절차**

**R**: `teams_filtered`에 `logRS = log(R)`, `logRA = log(RA)`를 추가하고
분석에 필요한 변수만 선택한 후 결측치를 제거하였다 (n = 450). 23개 설명변수를
포함한 절편 있는 full 로지스틱 회귀 모형을 적합한 후, `MASS::stepAIC()`로
양방향 단계적 변수 선택을 수행하였다. 선택된 모형에서 유의하지 않은 변수
(`CG`, `SHO`)를 추가로 제거하여 최종 모형을 확정하였다.

**Python**: `teams_filtered`에 `logRS`, `logRA`를 추가한 후 필요한 컬럼만
선택하고 결측치를 제거하였다. `statsmodels`에는 `stepAIC`에 해당하는 내장
함수가 없으므로, full model에서 출발하여 AIC가 감소하는 방향으로 변수를
추가/제거하는 양방향 단계적 선택 함수를 직접 구현하였다. 이후 R과 동일하게
`CG`, `SHO`를 제거한 최종 모형을 적합하였다.

------------------------------------------------------------------------

::: panel-tabset
## R

```{r}
library(MASS)
library(tidyverse)

# 1. 데이터 랭글링 (변수명 변경 및 결측치 제거)
teams_full <- teams_filtered %>%
  mutate(
    logRS = log(R),
    logRA = log(RA)
  )%>%
  dplyr::select(W, L, G, WPct, logRS, logRA, H, X2B, X3B, HR, BB, SO, CS, 
        HBP, SF, ERA, CG, SHO, IPouts, HA, HRA, BBA, SOA, E, DP, FP, SV) %>%
  drop_na()

# 2. 절편이 있는 Full 로지스틱 회귀 모형 적합
full_model <- glm(
  cbind(W, L) ~ logRS + logRA + H + X2B + X3B + HR + BB + SO + CS + 
                HBP + SF + ERA + CG + SHO + IPouts + HA + HRA + BBA + 
                SOA + E + DP + FP + SV,
  data = teams_full,
  family = binomial
)

# 3. AIC 기준 단계별 변수 선택 (양방향)
# trace = FALSE로 설정하여 중간 과정 출력 생략, anova로 요약 확인
step_model <- stepAIC(full_model, direction = "both", trace = FALSE)

# 4. 최종 선택된 모형 결과 확인
summary(step_model)

# 5. 'CG'와 'SHO' 최종 제거 모형
logit_model_final <- glm(
  cbind(W, L) ~ logRS + logRA + SV, 
  data =  teams_full, 
  family = binomial
)

# 회귀계수 추정치 및 요약 결과 확인
summary(logit_model_final)

```

## Python

In [ ]:
import polars as pl
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm

# 1. 데이터 랭글링 (변수명 변경 및 결측치 제거)
# 이전 단계의 teams_logistic_py_sep를 기반으로 진행
teams_full_py = (
    teams_filtered
    .with_columns(
        logRS = np.log(pl.col("R")),
        logRA = np.log(pl.col("RA"))
    )
)

predictors = [
    "logRS", "logRA", "H", "X2B", "X3B", "HR", "BB", "SO", "CS", 
    "HBP", "SF", "ERA", "CG", "SHO", "IPouts", "HA", "HRA", "BBA", 
    "SOA", "E", "DP", "FP", "SV"
]


# 분석에 쓰일 컬럼만 남기고 결측치 제거
columns_to_keep = ["WPct", "G"] + predictors
teams_pd_full = teams_full_py.select(columns_to_keep).drop_nulls().to_pandas()


# 2. 양방향 단계적 선택법(Bidirectional Stepwise Selection) 함수
def stepwise_selection_aic(df, target, predictors, weight_col):
    # R의 stepAIC(full_model)처럼 모든 변수를 포함한 상태에서 시작
    current_predictors = list(predictors)
    
    while True:
        # 현재 모형의 AIC 계산
        formula = f"{target} ~ 1 + " + " + ".join(current_predictors)
        current_model = smf.glm(formula=formula, data=df, family=sm.families.Binomial(), var_weights=df[weight_col]).fit()
        best_aic = current_model.aic
        
        best_action = None
        best_candidate = None
        
        # [Step A] Backward: 현재 모형에서 변수 하나씩 빼보기
        for var in current_predictors:
            test_preds = [p for p in current_predictors if p != var]
            test_formula = f"{target} ~ 1 + " + " + ".join(test_preds)
            test_model = smf.glm(formula=test_formula, data=df, family=sm.families.Binomial(), var_weights=df[weight_col]).fit()
            
            if test_model.aic < best_aic:
                best_aic = test_model.aic
                best_action = 'drop'
                best_candidate = var
                
        # [Step B] Forward: 현재 빠져있는 변수 하나씩 다시 넣어보기
        excluded_predictors = [p for p in predictors if p not in current_predictors]
        for var in excluded_predictors:
            test_preds = current_predictors + [var]
            test_formula = f"{target} ~ 1 + " + " + ".join(test_preds)
            test_model = smf.glm(formula=test_formula, data=df, family=sm.families.Binomial(), var_weights=df[weight_col]).fit()
            
            if test_model.aic < best_aic:
                best_aic = test_model.aic
                best_action = 'add'
                best_candidate = var
                
        # [Step C] AIC 개선 여부에 따라 행동 결정
        if best_action == 'drop':
            current_predictors.remove(best_candidate)
            print(f"Dropped: {best_candidate:<10} | New AIC: {best_aic:.4f}")
        elif best_action == 'add':
            current_predictors.append(best_candidate)
            print(f"Added:   {best_candidate:<10} | New AIC: {best_aic:.4f}")
        else:
            # 빼거나 넣어도 AIC가 더 이상 낮아지지 않으면 반복 종료
            print("\nStepwise selection completed.")
            break 
            
    # 최종 모형 적합
    final_formula = f"{target} ~ 1 + " + " + ".join(current_predictors)
    final_model = smf.glm(formula=final_formula, data=df, family=sm.families.Binomial(), var_weights=df[weight_col]).fit()
    return current_predictors, final_model
  
# 3. 함수 실행 및 최종 모형 확인
final_vars, step_model_py = stepwise_selection_aic(teams_pd_full, "WPct", predictors, "G")
print(step_model_py.summary())


# 5. 'CG'와 'SHO' 최종 제거 모형
logit_model_final = smf.glm(
    formula="WPct ~ logRS + logRA + SV",
    data=teams_pd_full,
    family=sm.families.Binomial(),
    var_weights=teams_pd_full["G"]
).fit()

# 회귀계수 추정치 및 요약 결과 확인
print(logit_model_final.summary())


:::

------------------------------------------------------------------------

#### **결과해석**

**단계적 변수 선택 결과 (stepAIC)**

23개 설명변수를 포함한 full model에서 출발하여 AIC 기준 양방향 단계적
변수 선택을 수행한 결과, `logRS`, `logRA`, `CG`, `SHO`, `SV` 5개 변수가
선택되었다 (AIC = 2601.4, Residual deviance = 106.12, df = 444).

**변수 추가 제거**

선택된 모형에서 `CG`(완투, p = 0.129)와 `SHO`(완봉, p = 0.065)는 통상적인
유의수준 $\alpha = 0.05$에서 유의하지 않으므로 추가 제거하였다. 최종 모형은
`logRS`, `logRA`, `SV`(세이브) 3개 변수와 절편으로 구성되며, AIC = 2603.5,
Residual deviance = 112.28 (df = 446)이다. stepAIC 모형 대비 AIC가 2.1
증가하였으나, 모형의 간결성을 고려할 때 적절한 선택이다.

**최종 모형 계수 해석**

- **`logRS`** ($\hat{\beta} = 1.614$, $p < 0.001$): 득점이 많을수록 승률이
  유의하게 높아진다.
- **`logRA`** ($\hat{\beta} = -1.441$, $p < 0.001$): 실점이 많을수록 승률이
  유의하게 낮아진다.
- **`SV`** ($\hat{\beta} = 0.011$, $p < 0.001$): 세이브 수가 많을수록 승률이
  유의하게 높아진다. 세이브는 불펜 투수진의 안정성을 반영하는 지표로,
  득실점 외에 승률을 추가적으로 설명한다.

**문제 1의 모형과 비교**

문제 2-1의 모형($\log(R/RA)$만 사용, $\hat{k} \approx 1.75$)과 비교하면,
최종 모형에서 `logRS`와 `logRA`의 계수가 각각 1.614와 -1.441로 절댓값이
서로 다르다. 이는 득점과 실점이 승률에 미치는 영향이 대칭적이지 않음을
시사하며, 단일 비율 $\log(RA/RS)$로 요약하는 피타고라스 모형의 가정이
완전히 성립하지 않을 수 있음을 보여준다. 또한 `SV`가 유의한 추가 예측변수로
선택되어, 득실점 외에 불펜 안정성도 팀 승률에 독립적인 기여를 함을 확인하였다.
Residual deviance도 문제 2-1-2의 180.38에서 112.28로 크게 감소하여 모형
적합도가 향상되었다.

#### **두 언어의 결과 비교 및 차이점**

선택된 변수(`logRS`, `logRA`, `CG`, `SHO`, `SV`)와 최종 모형의 회귀계수,
Residual deviance가 R과 Python에서 일치하였다. 단, stepwise 과정에서
표시되는 AIC 값이 크게 다른데(R: 2601.4 vs Python: 65037.2), 이는 추정
방식의 차이가 아니라 `statsmodels`가 `var_weights` 사용 시 AIC를 가중
log-likelihood 기반으로 계산하기 때문이다. 변수 선택의 기준은 AIC의 상대적
증감이므로 최종 선택 결과에는 영향을 미치지 않는다.

## 문제 2-3

1.  `W`(승리 횟수)를 반응변수로 하여 문제 2-2의 분석을 실시하되 포아송 회귀모형을 사용하라. 결과를 문제 2-2의 모형과 비교하라. 

2.  `W`를 반응변수로 하여 문제2의 분석을 실시하되 음이항 회귀모형을 사용하라. 모형 적합 시 오류가 발생하면 이유를 파악해서 보고하라.

### 답안(2-3-1)

#### **데이터 랭글링 절차**

**R**: 문제 2-2에서 생성한 `teams_full`을 그대로 사용하였다. 반응변수를
승리 횟수(`W`)로 변경하고, 팀마다 경기 수(`G`)가 다를 수 있으므로
`offset(log(G))`를 추가하여 노출(exposure)을 보정한 포아송 회귀 모형을
적합하였다. `stepAIC()`로 양방향 변수 선택 후, 유의하지 않은 `SHO`(p =
0.075)를 추가 제거하여 최종 모형을 확정하였다.

**Python**: `teams_full_py`에서 반응변수 `W`와 예측변수들을 선택한 후
결측치를 제거하였다. 2-2의 stepwise 함수를 포아송 회귀에 맞게 수정하여
`smf.glm(..., family=Poisson(), exposure=df[G])`를 사용하였으며, R과
동일하게 `SHO`를 추가 제거한 최종 모형을 적합하였다.

------------------------------------------------------------------------

::: panel-tabset
## R

```{r}
# 1. 절편이 있는 Full 포아송 회귀 모형 적합
# 반응변수 W, offset으로 log(G) 추가
full_poisson <- glm(
  W ~ logRS + logRA + H + X2B + X3B + HR + BB + SO + CS + 
      HBP + SF + ERA + CG + SHO + IPouts + HA + HRA + BBA + 
      SOA + E + DP + FP + SV + offset(log(G)),
  data = teams_full,
  family = poisson
)

# 2. AIC 기준 단계별 변수 선택 (양방향)
step_poisson <- stepAIC(full_poisson, direction = "both", trace = FALSE)

# 3. 최종 선택된 포아송 모형 결과 확인
summary(step_poisson)

# 4. SHO 변수 제거하고 최종 모형
final_poisson <- glm(
  W ~ logRS + ERA + SV + offset(log(G)),
  data = teams_full,
  family = poisson
)
summary(final_poisson)

```

## Python

In [ ]:
# 1. 포아송 분석을 위해 반응변수 'W'가 포함된 데이터셋을 다시 정의
columns_to_keep_poi = ["W", "G"] + predictors
teams_pd_poisson = teams_full_py.select(columns_to_keep_poi).drop_nulls().to_pandas()

# 2. 포아송 회귀용 양방향 단계적 선택법 함수 (family와 offset 처리 변경)
def stepwise_selection_poisson(df, target, predictors, exposure_col):
    current_predictors = list(predictors)
    
    while True:
        formula = f"{target} ~ 1 + " + " + ".join(current_predictors)
        # Poisson family와 exposure 인자 사용
        current_model = smf.glm(formula=formula, data=df, family=sm.families.Poisson(), exposure=df[exposure_col]).fit()
        best_aic = current_model.aic
        
        best_action = None
        best_candidate = None
        
        # [Step A] Backward
        for var in current_predictors:
            test_preds = [p for p in current_predictors if p != var]
            test_formula = f"{target} ~ 1 + " + " + ".join(test_preds)
            test_model = smf.glm(formula=test_formula, data=df, family=sm.families.Poisson(), exposure=df[exposure_col]).fit()
            
            if test_model.aic < best_aic:
                best_aic = test_model.aic
                best_action = 'drop'
                best_candidate = var
                
        # [Step B] Forward
        excluded_predictors = [p for p in predictors if p not in current_predictors]
        for var in excluded_predictors:
            test_preds = current_predictors + [var]
            test_formula = f"{target} ~ 1 + " + " + ".join(test_preds)
            test_model = smf.glm(formula=test_formula, data=df, family=sm.families.Poisson(), exposure=df[exposure_col]).fit()
            
            if test_model.aic < best_aic:
                best_aic = test_model.aic
                best_action = 'add'
                best_candidate = var
                
        # [Step C] AIC 개선 여부에 따라 행동 결정
        if best_action == 'drop':
            current_predictors.remove(best_candidate)
            print(f"Dropped: {best_candidate:<10} | New AIC: {best_aic:.4f}")
        elif best_action == 'add':
            current_predictors.append(best_candidate)
            print(f"Added:   {best_candidate:<10} | New AIC: {best_aic:.4f}")
        else:
            print("\nStepwise selection completed.")
            break 
            
    final_formula = f"{target} ~ 1 + " + " + ".join(current_predictors)
    final_model = smf.glm(formula=final_formula, data=df, family=sm.families.Poisson(), exposure=df[exposure_col]).fit()
    return current_predictors, final_model

# 3. 함수 실행 및 최종 모형 확인
final_vars_poi, step_model_poi = stepwise_selection_poisson(teams_pd_poisson, "W", predictors, "G")
print(step_model_poi.summary())

# 5. 'CG'와 'SHO' 최종 제거 모형
poisson_final = smf.glm(
    formula="W ~ logRS + ERA + SV",
    data=teams_pd_poisson,
    family=sm.families.Poisson(),
    exposure=teams_pd_poisson["G"]
).fit()

# 회귀계수 추정치 및 요약 결과 확인
print(poisson_final.summary())

:::

------------------------------------------------------------------------

#### **결과해석**

**단계적 변수 선택 결과**

포아송 회귀에서 AIC 기준 양방향 단계적 변수 선택 결과, `logRS`, `ERA`,
`SHO`, `SV` 4개 변수가 선택되었다 (AIC = 2874.2, Residual deviance =
64.37, df = 445). `SHO`(완봉)는 p = 0.075로 유의하지 않아 추가 제거하였다.

**최종 모형 결과**

최종 모형은 `logRS`, `ERA`, `SV`와 절편으로 구성된다 (AIC = 2875.4,
Residual deviance = 67.53, df = 446). 모든 변수가 유의하였다($p < 0.001$).

- **`logRS`** ($\hat{\beta} = 0.801$): 득점이 많을수록 승리 횟수가 증가한다.
- **`ERA`** ($\hat{\beta} = -0.164$): 평균자책점이 높을수록 승리 횟수가
  감소한다. ERA는 투수 성적의 핵심 지표로, 실점 억제 능력을 직접 반영한다.
- **`SV`** ($\hat{\beta} = 0.005$): 세이브 수가 많을수록 승리 횟수가
  증가한다.

**문제 2-2의 로지스틱 모형과 비교**

두 모형 모두 `logRS`와 `SV`를 공통 변수로 선택하였다. 그러나 로지스틱
모형은 `logRA`(실점의 로그)를 선택한 반면, 포아송 모형은 `ERA`(평균자책점)를
선택하였다. 두 변수 모두 팀의 실점 억제 능력을 반영하나, 포아송 모형에서는
`logRA` 대신 경기당 실점을 직접 표현하는 `ERA`가 더 설명력이 높게 나타났다.
또한 로지스틱 모형의 Residual deviance(112.28)보다 포아송 모형(67.53)이
더 작아 데이터에 더 잘 적합되었음을 보여준다.

#### **두 언어의 결과 비교 및 차이점**

R과 Python의 stepwise 선택 변수, 최종 모형의 회귀계수, 표준오차, Residual
deviance가 모두 일치하였다. 2-2와 달리 포아송 회귀에서는 Python의 AIC도
R과 동일한 스케일(2874-2875)로 계산되어 직접 비교가 가능하다. 이는
`var_weights` 사용 시에만 AIC 스케일 차이가 발생하고, `exposure` 인자
사용 시에는 R과 동일한 방식으로 계산되기 때문이다.

### 답안(2-3-2)

#### **데이터 랭글링 절차**

**R**: 문제 2-3-1의 `teams_full`을 그대로 사용하였다. `MASS::glm.nb()`로
음이항 회귀 full 모형 적합을 시도하였으며, `tryCatch()`로 오류 및 경고를
포착하였다.

**Python**: 포아송 회귀와 동일한 데이터(`teams_pd_poisson`)를 사용하였다.
`statsmodels.formula.api.negativebinomial()`로 음이항 회귀를 시도하였으며,
`warnings.catch_warnings()`로 경고를 오류로 변환하여 포착하였다.


------------------------------------------------------------------------

::: panel-tabset
## R

```{r}
library(MASS)
library(tidyverse)

# 음이항 회귀를 이용한 전체 분석(Full 모형 및 단계별 선택) 시도
nb_analysis_r <- tryCatch({
  
  full_nb_model <- glm.nb(
    W ~ logRS + logRA + H + X2B + X3B + HR + BB + SO + CS + 
        HBP + SF + ERA + CG + SHO + IPouts + HA + HRA + BBA + 
        SOA + E + DP + FP + SV + offset(log(G)),
    data = teams_full
  )
  
  step_nb_model <- stepAIC(full_nb_model, direction = "both", trace = FALSE)
  summary(step_nb_model)
  
}, error = function(e) {
  return(paste("오류 발생:", e$message))
}, warning = function(w) {
  return(paste("경고 발생:", w$message))
})

# 포착된 에러 또는 경고 메시지 출력
print(nb_analysis_r)
```

## Python

In [ ]:
import statsmodels.formula.api as smf
import warnings

def stepwise_selection_nb(df, target, predictors, exposure_col):
    current_predictors = list(predictors)
    
    formula = f"{target} ~ 1 + " + " + ".join(current_predictors)
    current_model = smf.negativebinomial(formula=formula, data=df, exposure=df[exposure_col]).fit(disp=0)
    best_aic = current_model.aic
    
    while True:
        best_action = None
        best_candidate = None
        
        for var in current_predictors:
            test_preds = [p for p in current_predictors if p != var]
            test_formula = f"{target} ~ 1 + " + " + ".join(test_preds)
            test_model = smf.negativebinomial(formula=test_formula, data=df, exposure=df[exposure_col]).fit(disp=0)
            
            if test_model.aic < best_aic:
                best_aic = test_model.aic
                best_action = 'drop'
                best_candidate = var
                
        excluded_predictors = [p for p in predictors if p not in current_predictors]
        for var in excluded_predictors:
            test_preds = current_predictors + [var]
            test_formula = f"{target} ~ 1 + " + " + ".join(test_preds)
            test_model = smf.negativebinomial(formula=test_formula, data=df, exposure=df[exposure_col]).fit(disp=0)
            
            if test_model.aic < best_aic:
                best_aic = test_model.aic
                best_action = 'add'
                best_candidate = var
                
        if best_action == 'drop':
            current_predictors.remove(best_candidate)
        elif best_action == 'add':
            current_predictors.append(best_candidate)
        else:
            break 
            
    final_formula = f"{target} ~ 1 + " + " + ".join(current_predictors)
    final_model = smf.negativebinomial(formula=final_formula, data=df, exposure=df[exposure_col]).fit(disp=0)
    return final_model

# 전체 과정 실행 시도 및 에러/경고 포착
try:
    # Python에서 발생하는 ConvergenceWarning 등의 경고를 에러로 취급하여 중단
    with warnings.catch_warnings():
        warnings.simplefilter("error") 
        step_model_nb = stepwise_selection_nb(teams_pd_poisson, "W", predictors, "G")
        print(step_model_nb.summary())
except Exception as e:
    print(f"경고 또는 오류 발생으로 중단됨: {e}")

:::

------------------------------------------------------------------------

#### **결과해석**

R과 Python 모두 음이항 회귀 모형 적합에 실패하였다.

- **R**: "iteration 제한에 도달했습니다" 경고 발생 — 최대 반복 횟수 내에
  알고리즘이 수렴하지 못하였음을 의미한다.
- **Python**: "divide by zero encountered in log" 오류 발생 — 로그 함수
  내부에 0 이하의 값이 입력되어 수치적으로 발산하였음을 의미한다.

**수렴 실패의 원인**

음이항 분포는 포아송 분포에 과산포(overdispersion) 모수 $\theta$를 추가한
모형으로, 데이터의 분산이 평균보다 클 때 포아송 회귀의 대안으로 사용된다.
그러나 이 데이터에서 수렴이 실패한 이유는 다음과 같다.

첫째, **과산포가 없는 데이터**이기 때문이다. 문제 2-3-1에서 포아송 모형의
Residual deviance = 67.53, df = 446으로 deviance/df ≈ 0.15에 불과하여
오히려 과소산포(underdispersion) 경향이 있다. 과산포가 없는 데이터에
음이항 모형을 적합하면 과산포 모수 $\theta$가 무한대로 발산하여 수렴에
실패하게 된다.

둘째, **설명변수의 수가 많아** 다중공선성이 심한 full 모형에서 최적화
알고리즘이 안정적으로 작동하지 못하는 수치적 불안정성이 발생할 수 있다.

따라서 이 데이터에서는 포아송 회귀가 적절한 모형이며, 음이항 회귀는
적합하지 않다.


#### **두 언어의 결과 비교 및 차이점**

R과 Python 모두 동일한 원인(수렴 실패)으로 모형 적합에 실패하였다.
오류 메시지의 형태는 다르나(R: 반복 제한 경고, Python: 로그 0 오류),
모두 과산포 모수 추정 과정에서의 수치적 불안정성을 반영한다.


## 문제 2-4

스테로이드 시대인 1994년에서 2005년의 기간과 최근 시대인 2010년에서 2025 기간의 $k$ 계수가 유의하게 변화하는지 파악하기 위해 $i$번째 팀과 연도 $t$에 대해 다음과 같은 식을 생각해 볼 수 있다.
$$
  WPct_(i,t)
  =
  \frac{1}{1+(RA_{i,t}/RS_{i,t} )^{k+g I(1994 \leq t \leq 2005)} }
$$
이 때 $I(1994 \leq t \leq 2005)$는 괄호안의 조건이 만족되면 1의 값을 가지고 아니면 0의 값을 가지는 지시함수이고, $g$는 스테로이드 시대와 최근 시대의 차이를 나타내는 계수이다. 위의 식에서 $g$가 0과 유의하게 같은지 가설검정을 수행하게 해주는 로지스틱 모형을 적합하고 결과를 해석하라. (코로나 시즌인 2020년은 제외한다.)


### 답안(2-4)

#### **데이터 랭글링 절차**

**R**: `Lahman` 패키지의 `Teams` 데이터에서 스테로이드 시대(1994–2005)와
최근 시대(2010–2025, 2020 제외)를 필터링하였다. `R`을 `RS`로 이름을
변경하고, 득실점 로그 비율(`log_ratio = log(RS/RA)`), 스테로이드 시대
지시함수(`is_steroid`), 승률(`WPct`)을 생성하였다. 절편 없는 로지스틱
회귀에 `log_ratio:is_steroid` 교호작용 항을 추가하여 시대별 $k$ 계수
차이($g$)를 추정하였다.

**Python**: `pylahman` 패키지가 2025년 데이터를 포함하지 않으므로 R
`Lahman` 패키지에서 내보낸 `Teams.csv`를 사용하였다. 동일한 필터링 및
변수 생성을 Polars로 수행한 후 `statsmodels`로 모형을 적합하였다.

------------------------------------------------------------------------

::: panel-tabset
## R

```{r}
library(tidyverse)
library(Lahman)

# 1. 데이터 랭글링 (두 시대 필터링 및 지시함수 변수 생성)
teams_era <- Teams %>%
  filter(
    (yearID >= 1994 & yearID <= 2005) | 
    (yearID >= 2010 & yearID <= 2025 & yearID != 2020)
  ) %>%
  rename(RS = R) %>%
  mutate(
    log_ratio = log(RS / RA),
    is_steroid = ifelse(yearID >= 1994 & yearID <= 2005, 1, 0),
    WPct = W / G
  ) %>%
  drop_na(W, G, log_ratio, is_steroid)

# 2. 로지스틱 회귀 모형 적합 (절편 없음, 교호작용 항 추가)
# log_ratio:is_steroid 는 log_ratio와 is_steroid를 곱한 교호작용(g)을 의미함
era_model_r <- glm(
  cbind(W, G - W) ~ 0 + log_ratio + log_ratio:is_steroid,
  data = teams_era,
  family = binomial
)

# 3. 회귀계수 추정치 및 가설검정 결과 확인
summary(era_model_r)
```

## Python

In [ ]:
import polars as pl
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm

# 1. 데이터 로드 및 랭글링
Teams = pl.read_csv(
    "data/Teams.csv",
    null_values=["NA", " ", "null"] )

teams_era_py = (
    Teams
    .filter(
        ((pl.col("yearID") >= 1994) & (pl.col("yearID") <= 2005)) |
        ((pl.col("yearID") >= 2010) & (pl.col("yearID") <= 2025) & (pl.col("yearID") != 2020))
    )
    .rename({"R": "RS"})
    .with_columns(
        log_ratio = np.log(pl.col("RS") / pl.col("RA")),
        # 스테로이드 시대이면 1, 아니면 0인 지시함수 변수 생성
        is_steroid = pl.when((pl.col("yearID") >= 1994) & (pl.col("yearID") <= 2005)).then(1).otherwise(0),
        WPct = pl.col("W") / pl.col("G")
    )
    .drop_nulls(subset=["WPct", "log_ratio", "is_steroid"])
)

teams_pd_era = teams_era_py.to_pandas()

# 2. 로지스틱 회귀 모형 적합 (절편 없음, 교호작용 항 추가)
era_model_py = smf.glm(
    formula="WPct ~ 0 + log_ratio + log_ratio:is_steroid",
    data=teams_pd_era,
    family=sm.families.Binomial(),
    var_weights=teams_pd_era["G"]
).fit()

# 3. 회귀계수 추정치 및 가설검정 결과 확인
print(era_model_py.summary())

:::

------------------------------------------------------------------------

#### **결과해석**

모형의 계수는 다음과 같이 해석된다.

- **`log_ratio`** ($\hat{k} = 1.753$, $p < 0.001$): 최근 시대(2010–2025)의
  피타고라스 지수로, 문제 2-1-1의 추정치와 동일하다.
- **`log_ratio:is_steroid`** ($\hat{g} = 0.161$, $p = 0.032$): 스테로이드
  시대의 추가적인 $k$ 증가분으로, 스테로이드 시대의 $k$는
  $1.753 + 0.161 = 1.914$로 추정된다.

귀무가설 $H_0: g = 0$ (두 시대의 $k$가 동일)에 대한 검정 결과,
$\hat{g} = 0.161$ (95% CI: [0.014, 0.307], $p = 0.032$)으로 유의수준
$\alpha = 0.05$에서 귀무가설을 기각한다. 즉, 스테로이드 시대(1994–2005)의
$k$가 최근 시대(2010–2025)보다 유의하게 크다. 이는 스테로이드 시대에는
득실점 비율이 승률을 더 강하게 설명하였음을 의미한다. 스테로이드
사용으로 타격 성적이 전반적으로 향상되어 득실점 격차가 커졌고, 이로 인해
피타고라스 지수가 높아진 것으로 해석할 수 있다.


#### **두 언어의 결과 비교 및 차이점**

R과 Python의 추정치(`log_ratio` = 1.7527, `log_ratio:is_steroid` = 0.1605),
표준오차, z값, p값, Residual deviance(315.80)가 완전히 일치하였다. 2-2와
동일하게 Python의 Log-Likelihood가 R보다 큰 음수로 표시되나, 이는
`var_weights` 사용 시 `statsmodels`의 AIC 계산 방식 차이에 의한 것으로
모형 추정 결과 자체에는 영향이 없다.


# 3부  데이터 분석 기술

숙제 2에서는 제출용 GitHub 저장소에 작업한 Quarto markdown 소스 파일(`hw02.qmd`)을 올리면 GitHub에서 자동으로 HTML 파일 및 주피터 노트북 파일(`.ipynb`)을 만들고 이것을 [GitHub Pages](https://docs.github.com/en/pages/quickstart)에서 웹페이지로 보이도록 설정하였다. 
여기서는 숙제 3 제출용 GibHub 저장소에 작업한 Quarto markdown 소스 파일(`hw03.qmd`)을 올리면 숙제 2에서의 작업 프로세스에 더해 자동 생성된 `.ipynb` 파일을 컨테이너화하여, GitHub에서 자동 생성된 컨테이너 이미지를 Binder 서비스를 이용하여 온라인에서 주피터 노트북 파일을 사용할 수 있도록 한다.

## 문제 3-1. Dockerfile 설정

로컬 저장소 최상위 디렉토리에 아래와 같은 `Dockerfile` 파일을 추가한다. 
```{yml}
# 1. 기반 이미지 설정
FROM rocker/tidyverse:4.4.0

# 2. 시스템 의존성 설치 (ImageMagick 포함)
USER root
RUN apt-get update && apt-get install -y \
    wget \
    git \
    imagemagick \
    libmagick++-dev \
    && rm -rf /var/lib/apt/lists/*

# 3. Miniconda 설치
ENV CONDA_DIR /opt/conda
RUN wget --quiet https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O ~/miniconda.sh && \
    /bin/bash ~/miniconda.sh -b -p /opt/conda && \
    rm ~/miniconda.sh

# 4. Conda 경로 설정 및 환경 생성
ENV PATH=$CONDA_DIR/bin:$PATH
RUN conda create -n r-reticulate python=3.10 -y && \
    conda install -n r-reticulate -c conda-forge numpy pandas matplotlib -y
# 추가로 필요한 패키지 설치

# 5. R 패키지 설치 (reticulate 및 필수 패키지)
RUN R -e "install.packages(c('reticulate', 'remotes', 'IRkernel'))" && \
    R -e "IRkernel::installspec(user = FALSE)"
# 추가로 필요한 패키지 설치

# 6. reticulate가 사용할 Python 경로 고정 (환경 변수)
ENV RETICULATE_PYTHON=/opt/conda/envs/r-reticulate/bin/python

# 7. Binder용 jovyan 유저 생성
ENV NB_USER=jovyan
ENV NB_UID=1000
RUN usermod -l ${NB_USER} rstudio && \
    usermod -d /home/${NB_USER} -m ${NB_USER} && \
    chown -R ${NB_USER} /opt/conda /home/${NB_USER}
    
# 8. 노트북 파일 복사
COPY _site/hw03.ipynb /home/${NB_USER}/hw03.ipynb
RUN chown ${NB_USER}:users /home/${NB_USER}/hw03.ipynb

USER ${NB_USER}
WORKDIR /home/${NB_USER}

# Binder가 기대하는 포트
EXPOSE 8888

```

### 답안

## 문제 3-2. GitHub Actions 워크플로우 수정

숙제 2에서 만들었던 `publish.yml`을 수정하여 기존의 배포 단계 끝에 Docker 컨테이너 이미지를 빌드하고 Github Container Registry (GHCR)에 푸시하는 단계를 추가한다.

```{yml}
# ... (기존 Quarto Render 단계 이후)

      - name: Log in to GitHub Container Registry
        uses: docker/login-action@v3
        with:
          registry: ghcr.io
          username: ${{ github.actor }}
          password: ${{ secrets.GITHUB_TOKEN }}

      - name: Build and push Docker image
        uses: docker/build-push-action@v5
        with:
          context: .
          push: true
          tags: ghcr.io/${{ github.repository_owner }}/my-r-env:latest
```

### 답안

## 문제 3-3. GitHub Pages에 Binder 링크 추가

GitHub Page를 사용하여 저장소를 웹페이지로 활용하는 부분은 숙제 2에서와 같다.

웹페이지에서 노트북을 내려받는 대신 [Binder](mybinder.org) 서비스를 이용하여 온라인으로 노트북을 실행할 수 있도록 위해 `README.md` 파일을 로컬 저장소 최상위 디렉토리에 다음과 같이 만들자.

```{markdown}
# 숙제 3

이름: [아무개]
학번: [나의 학번]

이 숙제의 상세 분석 결과는 아래 링크에서 확인하실 수 있습니다.

* [분석 리포트 (HTML)](./hw03.html) 
* [주피터 노트북 (ipynb)](https://mybinder.org/v2/gh/<유저명>/snu-stat/<repo명>/gh-pages?filepath=hw03.ipynb
```

여기서 `<유저명>`은 제출자의 GitHub 유저 아이디이며, `<repo명>`은 hw3-로 시작하는 제출자의 repository 이름이다.

작업을 GitHub 원격 저장소로 push한 후 숙제 2 문제 3-3의 3, 4번 과정을 반복하라.

### 답안
![그림1](image/1.png)
![그림2](image/2.png)
![그림3](image/3.png)
![그림4](image/4.png)
